# Reddit Mental Health Analysis using NLP and Machine Learning
## University Course Project: NLP and Unsupervised Machine Learning (Expanded)

**Author:** Student Researcher
**Date:** May 2026

### 1. Introduction
Mental health issues are increasingly discussed on social media platforms like Reddit. Analyzing these discussions provides insights into the prevalent themes and emotional states of users.

### 2. Objectives
*   Build an NLP pipeline for text preprocessing.
*   Discover hidden thematic clusters (Unsupervised).
*   Train classifiers (SVM, RF, LSTM) to predict mental health categories (Supervised).
*   Implement an interactive analysis prompt.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
import warnings
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from gensim import corpora, models
from textblob import TextBlob

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
import os
base_path = 'archive-2/Original Reddit Data/Labelled Data/'
files = ['LD DA 1.csv', 'LD EL1.csv', 'LD PF1.csv', 'LD TS 1.csv']
dfs = [pd.read_csv(os.path.join(base_path, f)) for f in files if os.path.exists(os.path.join(base_path, f))]
df = pd.concat(dfs, ignore_index=True).dropna(subset=['selftext'])
df['full_text'] = df['title'].fillna('') + " " + df['selftext']
print(f'Dataset Shape: {df.shape}')

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = nltk.word_tokenize(text)
    return " ".join([lemmatizer.lemmatize(t) for t in tokens if t not in stop_words])

df['cleaned_text'] = df['full_text'].apply(clean_text)

In [ ]:
tfidf = TfidfVectorizer(max_features=2000, min_df=5, max_df=0.7)
tfidf_matrix = tfidf.fit_transform(df['cleaned_text'])

## Section 7: Unsupervised Clustering (K-Means)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)
print("Top words per cluster:")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = tfidf.get_feature_names_out()
for i in range(5):
    print(f"Cluster {i}: {' '.join([terms[ind] for ind in order_centroids[i, :10]])}")

## Section 14: Supervised Learning (SVM, RF, LSTM)
We'll now train models to predict the `Label` column. I've added dynamic label handling to avoid errors if some rare categories are missing from the test set.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['Label'])
X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, y, test_size=0.2, random_state=42)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
labels_rf = np.unique(np.concatenate((y_test, y_pred_rf)))
print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf, labels=labels_rf, target_names=le.classes_[labels_rf]))

# SVM
svm = SVC(kernel='linear', probability=True, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
labels_svm = np.unique(np.concatenate((y_test, y_pred_svm)))
print("\nSVM Results:")
print(classification_report(y_test, y_pred_svm, labels=labels_svm, target_names=le.classes_[labels_svm]))

### Deep Learning (LSTM)

In [ ]:
# 1. Uninstall the broken versions
!pip uninstall -y tensorflow keras tensorflow-intel

# 2. Reinstall the compatible versions
!pip install tensorflow keras --no-cache-dir


In [ ]:
# import tensorflow as tf
# Try standard keras first, then fallback to tensorflow.keras
try:
    from keras.preprocessing.text import Tokenizer
    from keras.preprocessing.sequence import pad_sequences
    from keras.models import Sequential
    from keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
except ImportError:
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D

print("Keras modules loaded successfully!")


tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['cleaned_text'])
X_pad = pad_sequences(tokenizer.texts_to_sequences(df['cleaned_text']), maxlen=100)

X_t_l, X_v_l, y_t_l, y_v_l = train_test_split(X_pad, y, test_size=0.2, random_state=42)

model = Sequential([
    Embedding(5000, 128, input_length=100),
    SpatialDropout1D(0.2),
    LSTM(64, dropout=0.2),
    Dense(len(le.classes_), activation='softmax')
])
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print("Training LSTM...")
model.fit(X_t_l, y_t_l, epochs=3, batch_size=32, verbose=1)

## Section 17: Interactive Mental Health Analysis Prompt

In [ ]:
def analyze_my_text():
    text = input("Enter Reddit post content: ")
    cleaned = clean_text(text)
    sentiment = TextBlob(text).sentiment.polarity
    pred = svm.predict(tfidf.transform([cleaned]))
    label = le.inverse_transform(pred)[0]
    
    print(f"\nRESULTS:")
    print(f"Predicted Theme: {label}")
    print(f"Sentiment Score: {sentiment:.2f} ({'Positive' if sentiment > 0 else 'Negative' if sentiment < 0 else 'Neutral'})")

# analyze_my_text() # Uncomment to run

## Conclusion
We've successfully combined Unsupervised and Supervised learning to analyze mental health discussions!